# Centaur IME: ベトナム語の AI（今の方法の候補を並べ替える形）

今の方法（辞書と語の並びの表、AI なし）が上位 8 個の候補を出し、AI はその文の「その言語らしさ」の点数を足して
並べ替えるだけ。重みは dev の文で決め、試験の文で 1 回だけ測る。重み 0 が今の方法と同じなので、悪くはならない。

使い方: 「ランタイム → ランタイムのタイプを変更」で **L4 GPU**、「ランタイム → すべてのセルを実行」。
**最初のセルで Google ドライブにつなぐ**（許可する）。結果はドライブ（CentaurIME/models/2026-11-07）に書き、最後にランタイムを手放す。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CentaurIME/models/2026-11-07'
import os; os.makedirs(OUT, exist_ok=True); print('保存先', OUT)

In [ ]:
!rm -rf repo && git clone -q --depth 1 -b work/lang-measure https://github.com/J-KAMMA/centaur-ime.git repo && git -C repo log -1 --format='%H %s'
!pip -q install pyarrow==21.0.0

In [ ]:
!python repo/tools/fetch_source.py tatoeba-vie data/vi
!python repo/tools/fetch_source.py wiki-vi data/vi
!python repo/tools/fetch_source.py cc100-vi data/vi --lines 400000

In [ ]:
# 今の方法で測り、上位の候補を書き出す
!python repo/tools/lang_measure/vi_measure.py --tatoeba data/vi/vie_sentences.tsv.bz2 --wiki $(ls data/vi/*.parquet | head -1) --out m --nbest-out nbest.jsonl
!cat m/*-measure.txt | head -12
!wc -l nbest.jsonl

In [ ]:
!python repo/tools/models/train_task.py --task rr-vi --tatoeba data/vi/vie_sentences.tsv.bz2 --cc100 data/vi/cc100-vi.400000.txt --extra 400000 --d 384 --layers 8 --heads 6 --batch 256 --minutes 40 --amp --nbest-file nbest.jsonl --out "$OUT"
print(open(OUT + '/rr-vi-result.txt', encoding='utf-8').read())

In [ ]:
from google.colab import runtime
runtime.unassign()